# Вычислительная ниткография

Алгоритм по портретной фотографии строт схему расположения нитей.
Метод основан на **Filtered Back Projection** из рентгеновской томографии.

In [ ]:
import numpy as np

from string_art import (
    load_grayscale_square,
    prepare_fbp_image,
    bilinear_sample,
    circle_mask,
    draw_threads,
)
from utils import show_image, show_side_by_side, show_sinogram, show_binary_mask, show_threads_on_black

## Исходное изображение

Загружаем, проверяем квадратную и одноканальную картинку.

In [ ]:
image = load_grayscale_square('input.png')
print(f'Размер: {image.shape[0]}×{image.shape[1]}, диапазон: [{image.min():.2f}, {image.max():.2f}]')

show_image(image, 'Исходное фото')

## Подготовка

Поднятием общей яркости исходного изображения устраняем отрицательные значения после рамп-фильтра.
Инвертируем, чтобы тёмные детали (глаза, волосы) получили больше нитей.

In [ ]:
target = prepare_fbp_image(image, brightness_lift=0.2)

show_side_by_side(image, target, 'До подготовки', 'После (инверсия + подъём яркости)')

## Шаг 1. Преобразование Радона

> *«$R[f(x,y)](\rho,\varphi) = \int_{-\infty}^{\infty} f(\rho\cos\varphi - l\sin\varphi,\; \rho\sin\varphi + l\cos\varphi)\,dl$»*

Для каждого угла $\varphi$ проводим набор параллельных прямых и суммируем яркость вдоль каждой.
Результат — **синограмма**: строки = углы, столбцы = смещения $\rho$.

In [ ]:
def radon_transform(image, angle_count):
    size = image.shape[0]
    center = (size - 1) / 2
    image = image * circle_mask(size)

    offsets = np.arange(size) - center
    line_points = np.linspace(-center, center, size)
    angles = np.linspace(0.0, 180.0, angle_count, endpoint=False)
    sinogram = np.zeros((angle_count, size), dtype=float)

    for angle_index, angle in enumerate(angles):
        phi = np.deg2rad(angle)
        cos_phi = np.cos(phi)
        sin_phi = np.sin(phi)

        x = offsets[:, None] * cos_phi - line_points[None, :] * sin_phi
        y = offsets[:, None] * sin_phi + line_points[None, :] * cos_phi

        rows = center + y
        cols = center + x
        sinogram[angle_index] = bilinear_sample(image, rows, cols).sum(axis=1)

    return sinogram, angles

In [ ]:
ANGLES = 360

sinogram, angles = radon_transform(target, ANGLES)
print(f'Синограмма: {sinogram.shape[0]} углов × {sinogram.shape[1]} смещений')

show_sinogram(sinogram, angles, 'Шаг 1: Синограмма — преобразование Радона',
              colorbar_label='Сумма яркости')

## Шаг 2. Свертка с рамп-фильтром

> *«$F_r(\rho)$ при $\rho=0$: $1/4$; при чётном $\rho$: $0$; при нечётном: $-1/(\pi\rho)^2$»*

Каждую проекцию сворачиваем с этим фильтром — подчёркиваем границы и детали.
После фильтрации могут появиться отрицательные значения.

In [ ]:
def ramp_kernel(size):
    offsets = np.arange(-(size // 2), size - size // 2)
    kernel = np.zeros(size, dtype=float)

    kernel[offsets == 0] = 1 / 4
    odd = (offsets % 2 != 0)
    kernel[odd] = -1 / (np.pi * offsets[odd]) ** 2

    return kernel


def filter_sinogram(sinogram):
    kernel = ramp_kernel(sinogram.shape[1])
    filtered = np.zeros_like(sinogram)

    for angle_index, projection in enumerate(sinogram):
        filtered[angle_index] = np.convolve(projection, kernel, mode="same")

    return filtered

In [ ]:
filtered = filter_sinogram(sinogram)
print(f'Минимум после фильтрации: {filtered.min():.2f} (отрицательные — нормально)')

show_sinogram(filtered, angles, 'Шаг 2: После рамп-фильтра (красный — отрицательные)',
              cmap='RdBu', colorbar_label='Значение',
              vmin=-np.abs(filtered).max(), vmax=np.abs(filtered).max())

## Шаг 3. Прореживание

Прореживание проекций по пространственной переменной до заданного числа нитей.

Из всех возможных прямых под каждым углом оставляем только **40 самых ярких**.
Остальные зануляем — нитей будет регулируемое количество.

In [ ]:
def keep_brightest_per_angle(sinogram, thread_count):
    kept = np.zeros_like(sinogram)
    thread_count = min(thread_count, sinogram.shape[1])

    for angle_index, projection in enumerate(sinogram):
        indices = np.argpartition(projection, -thread_count)[-thread_count:]
        kept[angle_index, indices] = projection[indices]

    return kept

In [ ]:
THREADS = 40

thinned = keep_brightest_per_angle(filtered, THREADS)
nonzero = (thinned > 0).sum()
total = thinned.size
print(f'Оставлено {nonzero} из {total} ячеек ({100*nonzero/total:.1f}%)')

show_binary_mask(thinned != 0, angles, f'Шаг 3: После прореживания (не более {THREADS} на угол)')

## Шаг 4. Нормировка

Занулим точки синограммы, значения которых меньше 0; нормируем оставшийся диапазон к отрезку $[0,1]$

Отрицательные → 0, остальные приводим к вероятностям от 0 до 1.
Теперь каждое число = вероятность того, что нить будет протянута.

In [ ]:
def normalize_sinogram_probabilities(sinogram, gamma=0.85):
    normalized = sinogram.copy()
    normalized[normalized < 0] = 0

    for angle_index, projection in enumerate(normalized):
        row = projection.copy()
        row[row < 0] = 0.0
        maximum = row.max()

        if maximum <= 0:
            continue

        row = (row / maximum) ** gamma
        normalized[angle_index] = row

    return normalized

In [ ]:
probabilities = normalize_sinogram_probabilities(thinned, gamma=0.85)
print(f'Диапазон вероятностей: [{probabilities.min():.3f}, {probabilities.max():.3f}]')

show_sinogram(probabilities, angles, 'Шаг 4: Вероятности (нормировка к [0, 1])',
              cmap='hot', colorbar_label='Вероятность нити', vmin=0, vmax=1)

## Шаг 5. Бинарное распыление

В каждом пикселе выходной синограммы поставим 1 с вероятностью, указанной в этом пикселе, иначе поставим 0

In [ ]:
def binary_dither(probabilities, seed):
    random = np.random.default_rng(seed).random(probabilities.shape)
    return (random < probabilities).astype(np.uint8)

In [ ]:
binary = binary_dither(probabilities, seed=1)
thread_count = binary.sum()
print(f'Нитей: {thread_count}')

show_binary_mask(binary, angles, f'Шаг 5: Бинарная синограмма ({thread_count} нитей)')

## Результат — обратная проекция

Из задания:

> *«Требуется построить обратную проекцию данной бинарной синограммы»*

Каждая единица в бинарной синограмме = одна нить. Протягиваем все нити и смотрим, что получилось.
Для контраста применяем *«контрастирование с зашкаливанием небольшого числа пикселей с высокой яркостью»*.

In [ ]:
def binary_sinogram_to_lines(binary_sinogram, angles, probabilities=None):
    center = (binary_sinogram.shape[1] - 1) / 2
    lines = []
    weights = []

    for angle_index, offset_index in np.argwhere(binary_sinogram > 0):
        angle = float(angles[angle_index])
        offset = float(offset_index - center)
        lines.append((angle, offset))
        if probabilities is not None:
            weights.append(float(probabilities[angle_index, offset_index]))
        else:
            weights.append(1.0)

    return lines, weights

In [ ]:
lines, weights = binary_sinogram_to_lines(binary, angles, probabilities)
size = image.shape[0]

canvas = draw_threads(lines, size, weights=weights)

# Контрастирование (quantile 0.995 — «зашкаливание» по заданию)
positive = canvas[canvas > 0]
limit = np.quantile(positive, 0.995) if positive.size > 0 else 1.0
preview = np.clip(canvas, 0, limit)
preview /= preview.max()

# Инверсия для отображения: нити -> тёмные (как на фото)
display = 1.0 - preview

show_side_by_side(image, display, 'Исходное фото', f'Результат ({thread_count} нитей)',
                    figsize=(11, 5))

## Белые нити на чёрном фоне

По заданию: *«будем строить ниткографическое изображение на черном фоне… нить одного цвета — белого»*.

Ниже — как выглядит реальная картина из ниток (без инверсии, нити белые).

In [ ]:
show_threads_on_black(preview, 'Белые нити на чёрном фоне (как в реальности)')